In [10]:
!export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/home/dongyoonhwang/.mujoco/mujoco210/bin:/usr/lib/nvidia
!export MUJOCO_PY_MJKEY_PATH=/home/nas2_userI/byungkunlee/.mujoco/mjkey.txt
!export MJKEY_PATH=/home/nas2_userI/byungkunlee/.mujoco/mjkey.txt
!export MJLIB_PATH=/home/nas2_userI/byungkunlee/.mujoco/mujoco210/bin/libmujoco210.so
!export MUJOCO_PY_MUJOCO_PATH=/home/dongyoonhwang/.mujoco/mujoco210

# >>> MuJoCo visualization >>>
!export XLA_PYTHON_CLIENT_PREALLOCATE=false
!export MUJOCO_GL='egl'
!export MUJOCO_EGL_DEVICE_ID='0'
!export MKL_SERVICE_FORCE_INTEL='0'

In [11]:
import sys
import os
import math
import pandas as pd
sys.path.append("../")
sys.path.append("../..")

In [12]:
from scale_rl.common.wandb_utils import *

In [13]:
td7_eval_df = pd.read_csv('../../results/baseline/td7.csv', index_col=0)
sac_simba_eval_df = pd.read_csv('../../results/baseline/simba.csv', index_col=0)
tdmpc2_eval_df = pd.read_csv('../../results/baseline/tdmpc2.csv', index_col=0)
dreamerv3_eval_df = pd.read_csv('../../results/baseline/dreamerv3.csv', index_col=0)
hypersimba_eval_df = pd.read_csv('../../results/simbaV2_utd2.csv', index_col=0)

sac_simba_eval_df['exp_name'] = 'Simba'
tdmpc2_eval_df['exp_name'] = 'TD-MPC2'
dreamerv3_eval_df['exp_name'] = 'DreamerV3'
hypersimba_eval_df['exp_name'] = 'SimbaV2'

#### Collection

In [14]:
off_policy_df = pd.concat([
    hypersimba_eval_df,
    sac_simba_eval_df, 
    td7_eval_df,
    tdmpc2_eval_df,
    dreamerv3_eval_df,
    ], ignore_index=True, sort=False)
off_policy_df['env_name'] = off_policy_df['env_name'].str.replace('_', '-')
off_policy_df

,exp_name,env_name,seed,metric,env_step,value
0,SimbaV2,h1-pole-v0,9000,avg_return,0.0,34.269159
1,SimbaV2,h1-pole-v0,9000,avg_return,100000.0,303.279434
2,SimbaV2,h1-pole-v0,9000,avg_return,200000.0,590.137001
3,SimbaV2,h1-pole-v0,9000,avg_return,300000.0,657.775993
4,SimbaV2,h1-pole-v0,9000,avg_return,400000.0,711.609390
...,...,...,...,...,...,...
67171,DreamerV3,walker-walk,3,avg_return,300000.0,925.014982
67172,DreamerV3,walker-walk,3,avg_return,350000.0,936.634629
67173,DreamerV3,walker-walk,3,avg_return,400000.0,792.896831
67174,DreamerV3,walker-walk,3,avg_return,450000.0,938.021900


In [15]:
exp_names = off_policy_df['exp_name'].unique()
exp_names

array(['SimbaV2', 'Simba', 'td7', 'TD-MPC2', 'DreamerV3'], dtype=object)

In [16]:
env_names = off_policy_df['env_name'].unique()
env_names

array(['h1-pole-v0', 'h1-slide-v0', 'h1-stair-v0', 'h1-balance-hard-v0',
       'h1-balance-simple-v0', 'h1-sit-hard-v0', 'h1-sit-simple-v0',
       'h1-maze-v0', 'h1-crawl-v0', 'h1-hurdle-v0', 'h1-reach-v0',
       'h1-run-v0', 'h1-stand-v0', 'h1-walk-v0', 'myo-pen-twirl-hard',
       'myo-pen-twirl', 'myo-key-turn-hard', 'myo-key-turn',
       'myo-obj-hold-hard', 'myo-obj-hold', 'myo-pose-hard', 'myo-pose',
       'myo-reach-hard', 'myo-reach', 'dog-trot', 'dog-run', 'dog-walk',
       'dog-stand', 'humanoid-run', 'humanoid-walk', 'humanoid-stand',
       'walker-run', 'walker-walk', 'walker-stand', 'reacher-hard',
       'reacher-easy', 'quadruped-run', 'quadruped-walk',
       'pendulum-swingup', 'hopper-stand', 'hopper-hop', 'fish-swim',
       'finger-turn-hard', 'finger-turn-easy', 'finger-spin',
       'cheetah-run', 'cartpole-swingup-sparse', 'cartpole-swingup',
       'cartpole-balance-sparse', 'cartpole-balance', 'ball-in-cup-catch',
       'acrobot-swingup', 'Humanoid-v4',

### Visualization

In [17]:
from rliable import library as rly
from rliable import metrics as rly_metrics
from rliable import plot_utils as rly_plot_utils

aggregate_func = lambda x: np.array([
  rly_metrics.aggregate_iqm(x),
  rly_metrics.aggregate_median(x),
  rly_metrics.aggregate_mean(x)])

In [18]:
from scale_rl.envs.mujoco import MUJOCO_ALL, MUJOCO_RANDOM_SCORE, MUJOCO_TD3_SCORE
from scale_rl.envs.dmc import DMC_EASY_MEDIUM, DMC_HARD
from scale_rl.envs.humanoid_bench import HB_LOCOMOTION_NOHAND, HB_RANDOM_SCORE, HB_SUCCESS_SCORE
from scale_rl.envs.myosuite import MYOSUITE_TASKS

def replace_hypen_to_underbar(env_name_list):
    for idx in range(len(env_name_list)):
        env_name_list[idx] = env_name_list[idx].replace('_', '-')
    return env_name_list
def replace_hypen_to_underbar_dict(env_name_dict):
    _new_dict = {}
    for k, v in env_name_dict.items():
        _new_dict[k.replace('_', '-')] = v
    return _new_dict

MUJOCO_ALL = replace_hypen_to_underbar(MUJOCO_ALL)
DMC_EASY_MEDIUM = replace_hypen_to_underbar(DMC_EASY_MEDIUM)
DMC_HARD = replace_hypen_to_underbar(DMC_HARD)
HB_LOCOMOTION_NOHAND = replace_hypen_to_underbar(HB_LOCOMOTION_NOHAND)
MYOSUITE_TASKS = replace_hypen_to_underbar(MYOSUITE_TASKS)

HB_SUCCESS_SCORE = replace_hypen_to_underbar_dict(HB_SUCCESS_SCORE)
HB_RANDOM_SCORE = replace_hypen_to_underbar_dict(HB_RANDOM_SCORE)

DMC_ALL = [*DMC_EASY_MEDIUM, *DMC_HARD]

In [19]:
MUJ_STEPS = 1000000 # 1M
DMC_STEPS = 1000000 # 1M
MYO_STEPS = 1000000 # 1M
HB_STEPS = 1000000 # 1M

In [20]:
def full_results_per_env(
    eval_df,
    total_env_steps,
    metric_matrix_dict,
    aggregate_func,
    metric: str = "avg_return",
    columns: List[str] = ["TD7", "Simba", "TD-MPC2", "DreamerV3", "SimbaV2"],
):
    experiments = eval_df["exp_name"].unique()
    assert sorted(columns) == sorted(experiments)
    environments = eval_df["env_name"].unique()
    
    aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
        metric_matrix_dict, aggregate_func, reps=10000
    )
    
    eval_df = eval_df[eval_df["metric"] == metric]
    eval_df = eval_df[eval_df["env_step"] <= total_env_steps]
    
    full_results_df = pd.DataFrame(columns=["Task"].append(columns))
    
    for _exp_name in experiments:
        _exp_data = eval_df[eval_df["exp_name"] == _exp_name]
        _num_seeds = _exp_data["seed"].nunique()
        print(f"exp_name: {_exp_name} - num_seeds: {_num_seeds}")
        
    for i, env in enumerate(environments):
        env_data = eval_df[eval_df["env_name"] == env]
        data = {"\\textbf{Task}": "\\texttt{" + str(env) + "}"}
        for j, exp in enumerate(columns):
            exp_data = env_data[env_data["exp_name"] == exp]
            if len(exp_data) == 0:
                continue
            # assert max(exp_data["env_step"]) == total_env_steps
            exp_data = exp_data[exp_data["env_step"] == max(exp_data["env_step"])]
            
            grouped_data = exp_data.groupby("env_step")["value"]

            env_steps = grouped_data.mean().index.values
            mean = float(grouped_data.mean().values)
            std_err = float(grouped_data.sem().values)
            low_CI = mean - 1.960 * std_err
            high_CI = mean + 1.960 * std_err
            
            if metric == "avg_success": 
                mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
            else:    
                mean, low_CI, high_CI = int(mean), int(low_CI), int(high_CI)
            
            
            data["\\textbf{" + f"{exp}" + "}"] = f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}"

        full_results_df = pd.concat([full_results_df, pd.DataFrame.from_dict(
                                data=[data], orient='columns')], ignore_index=True)

    
    for i, agg in enumerate(["IQM", "Median", "Mean"]):
        data = {"\\textbf{Task}": agg}
        for j, exp in enumerate(columns):
            mean = aggregate_scores[exp][i] 
            low_CI = aggregate_score_cis[exp][0][i] 
            high_CI = aggregate_score_cis[exp][1][i]
            
            if metric == "avg_success": 
                mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
            else:
                mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)
        
            data["\\textbf{" + f"{exp}" + "}"] = f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}"
        
        full_results_df = pd.concat([full_results_df, pd.DataFrame.from_dict(
                            data=[data], orient='columns')], ignore_index=True)

    return full_results_df
        

#### MuJoCo (Simba 1M) 

In [21]:
sac_simba_1m_eval_df = pd.read_csv('/home/nas4_user/youngdolee/SimbaV2_dev_imp/results/baseline/simba_utd2.csv', index_col=0)

sac_simba_mujuco_eval_df = sac_simba_1m_eval_df[sac_simba_1m_eval_df['env_name'].isin(MUJOCO_ALL)].sort_values("env_name")
total_env_steps = MUJ_STEPS
metric = "avg_return"
metric_matrix_dict = generate_metric_matrix_dict(
    normalize_score_with_random_and_base_score(
        df=sac_simba_mujuco_eval_df, 
        random_score_dict=MUJOCO_RANDOM_SCORE,
        base_score_dict=MUJOCO_TD3_SCORE,
    ),
    env_step=total_env_steps, 
    metric_type=metric,
)

In [22]:
experiments = sac_simba_mujuco_eval_df["exp_name"].unique()
environments = sac_simba_mujuco_eval_df["env_name"].unique()

metric_matrix_dict = generate_metric_matrix_dict(
    normalize_score_with_random_and_base_score(
        df=sac_simba_mujuco_eval_df, 
        random_score_dict=MUJOCO_RANDOM_SCORE,
        base_score_dict=MUJOCO_TD3_SCORE,
    ),
    env_step=total_env_steps, 
    metric_type=metric,
)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    metric_matrix_dict, aggregate_func, reps=10000
)

sac_simba_mujuco_eval_df = sac_simba_mujuco_eval_df[sac_simba_mujuco_eval_df["metric"] == metric]
sac_simba_mujuco_eval_df = sac_simba_mujuco_eval_df[sac_simba_mujuco_eval_df["env_step"] <= total_env_steps]

mujoco_full_results_df = pd.DataFrame(columns=["Task", "HyperSimba"])

for _exp_name in experiments:
    _exp_data = sac_simba_mujuco_eval_df[sac_simba_mujuco_eval_df["exp_name"] == _exp_name]
    _num_seeds = _exp_data["seed"].nunique()
    print(f"exp_name: {_exp_name} - num_seeds: {_num_seeds}")
    
for i, env in enumerate(environments):
    mean_ci = []
    env_data = sac_simba_mujuco_eval_df[sac_simba_mujuco_eval_df["env_name"] == env]
    for j, exp in enumerate(experiments):
        exp_data = env_data[env_data["exp_name"] == exp]
        if len(exp_data) == 0:
            continue
        assert max(exp_data["env_step"]) == total_env_steps
        exp_data = exp_data[exp_data["env_step"] == total_env_steps]
        num_seeds = exp_data["seed"].nunique()
        
        grouped_data = exp_data.groupby("env_step")["value"]

        env_steps = grouped_data.mean().index.values
        mean = float(grouped_data.mean().values)
        std_dev = float(grouped_data.std().values)

        # Plot mean history
        low_CI = mean - 1.960 * std_dev / math.sqrt(num_seeds)
        high_CI = mean + 1.960 * std_dev / math.sqrt(num_seeds)
        
        if metric == "avg_success":
            mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
        else:    
            mean, low_CI, high_CI = int(mean), int(low_CI), int(high_CI)
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    mujoco_full_results_df = pd.concat([mujoco_full_results_df, pd.DataFrame.from_dict(
                            data=[{"\\textbf{Task}": "\\texttt{" + str(env) + "}",
                            "Simba": mean_ci[0],
                            }], orient='columns')], ignore_index=True)


for i, agg in enumerate(["IQM", "Median", "Mean"]):
    mean_ci = []
    for j, exp in enumerate(experiments):
        mean = aggregate_scores[exp][i] 
        low_CI = aggregate_score_cis[exp][0][i] 
        high_CI = aggregate_score_cis[exp][1][i]

        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)
        
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    
    mujoco_full_results_df = pd.concat([mujoco_full_results_df, pd.DataFrame.from_dict(
                        data=[{"Task": agg,
                        "Simba": mean_ci[0],
                        }], orient='columns')], ignore_index=True)

mujoco_full_results_df

exp_name: sac_simba - num_seeds: 10


,Task,HyperSimba,\textbf{Task},Simba
0,NaN,NaN,\texttt{Ant-v4},"5882 \textcolor{gray}{[5354, 6411]}"
1,NaN,NaN,\texttt{HalfCheetah-v4},"9422 \textcolor{gray}{[8745, 10100]}"
2,NaN,NaN,\texttt{Hopper-v4},"3231 \textcolor{gray}{[3004, 3458]}"
3,NaN,NaN,\texttt{Humanoid-v4},"6513 \textcolor{gray}{[5634, 7392]}"
4,NaN,NaN,\texttt{Walker2d-v4},"4290 \textcolor{gray}{[3864, 4716]}"
5,IQM,NaN,NaN,"1.114 \textcolor{gray}{[1.045, 1.198]}"
6,Median,NaN,NaN,"1.143 \textcolor{gray}{[1.063, 1.228]}"
7,Mean,NaN,NaN,"1.147 \textcolor{gray}{[1.076, 1.223]}"


In [23]:
sac_simba_1m_eval_df = pd.read_csv('../../results/1219_sac_simba_1m.csv', index_col=0)

sac_simba_mujuco_eval_df = sac_simba_1m_eval_df[sac_simba_1m_eval_df['env_name'].isin(MUJOCO_ALL)].sort_values("env_name")
total_env_steps = MUJ_STEPS
metric = "avg_return"
metric_matrix_dict = generate_metric_matrix_dict(
    normalize_score_with_random_and_base_score(
        df=sac_simba_mujuco_eval_df, 
        random_score_dict=MUJOCO_RANDOM_SCORE,
        base_score_dict=MUJOCO_TD3_SCORE,
    ),
    env_step=total_env_steps, 
    metric_type=metric,
)

In [24]:
experiments = sac_simba_mujuco_eval_df["exp_name"].unique()
environments = sac_simba_mujuco_eval_df["env_name"].unique()

metric_matrix_dict = generate_metric_matrix_dict(
    normalize_score_with_random_and_base_score(
        df=sac_simba_mujuco_eval_df, 
        random_score_dict=MUJOCO_RANDOM_SCORE,
        base_score_dict=MUJOCO_TD3_SCORE,
    ),
    env_step=total_env_steps, 
    metric_type=metric,
)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    metric_matrix_dict, aggregate_func, reps=10000
)

sac_simba_mujuco_eval_df = sac_simba_mujuco_eval_df[sac_simba_mujuco_eval_df["metric"] == metric]
sac_simba_mujuco_eval_df = sac_simba_mujuco_eval_df[sac_simba_mujuco_eval_df["env_step"] <= total_env_steps]

mujoco_full_results_df = pd.DataFrame(columns=["Task", "Simba"])

for _exp_name in experiments:
    _exp_data = sac_simba_mujuco_eval_df[sac_simba_mujuco_eval_df["exp_name"] == _exp_name]
    _num_seeds = _exp_data["seed"].nunique()
    print(f"exp_name: {_exp_name} - num_seeds: {_num_seeds}")
    
for i, env in enumerate(environments):
    mean_ci = []
    env_data = sac_simba_mujuco_eval_df[sac_simba_mujuco_eval_df["env_name"] == env]
    for j, exp in enumerate(experiments):
        exp_data = env_data[env_data["exp_name"] == exp]
        if len(exp_data) == 0:
            continue
        assert max(exp_data["env_step"]) == total_env_steps
        exp_data = exp_data[exp_data["env_step"] == total_env_steps]
        num_seeds = exp_data["seed"].nunique()
        
        grouped_data = exp_data.groupby("env_step")["value"]

        env_steps = grouped_data.mean().index.values
        mean = float(grouped_data.mean().values)
        std_dev = float(grouped_data.std().values)

        # Plot mean history
        low_CI = mean - 1.960 * std_dev / math.sqrt(num_seeds)
        high_CI = mean + 1.960 * std_dev / math.sqrt(num_seeds)
        
        if metric == "avg_success":
            mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
        else:    
            mean, low_CI, high_CI = int(mean), int(low_CI), int(high_CI)
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    mujoco_full_results_df = pd.concat([mujoco_full_results_df, pd.DataFrame.from_dict(
                            data=[{"\\textbf{Task}": "\\texttt{" + str(env) + "}",
                            "Simba": mean_ci[0],
                            }], orient='columns')], ignore_index=True)


for i, agg in enumerate(["IQM", "Median", "Mean"]):
    mean_ci = []
    for j, exp in enumerate(experiments):
        mean = aggregate_scores[exp][i] 
        low_CI = aggregate_score_cis[exp][0][i] 
        high_CI = aggregate_score_cis[exp][1][i]

        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)
        
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    
    mujoco_full_results_df = pd.concat([mujoco_full_results_df, pd.DataFrame.from_dict(
                        data=[{"Task": agg,
                        "Simba": mean_ci[0],
                        }], orient='columns')], ignore_index=True)

mujoco_full_results_df

exp_name: sac_simba - num_seeds: 10


,Task,Simba,\textbf{Task}
0,NaN,"6544 \textcolor{gray}{[5988, 7100]}",\texttt{Ant-v4}
1,NaN,"10461 \textcolor{gray}{[10081, 10841]}",\texttt{HalfCheetah-v4}
2,NaN,"3101 \textcolor{gray}{[2474, 3728]}",\texttt{Hopper-v4}
3,NaN,"7654 \textcolor{gray}{[6724, 8583]}",\texttt{Humanoid-v4}
4,NaN,"4685 \textcolor{gray}{[4513, 4858]}",\texttt{Walker2d-v4}
5,IQM,"1.219 \textcolor{gray}{[1.114, 1.35]}",NaN
6,Median,"1.289 \textcolor{gray}{[1.159, 1.363]}",NaN
7,Mean,"1.256 \textcolor{gray}{[1.161, 1.349]}",NaN


In [25]:
mujoco_latex_df = mujoco_full_results_df.to_latex(index=False)
mujoco_latex_df

'\\begin{tabular}{lll}\n\\toprule\nTask & Simba & \\textbf{Task} \\\\\n\\midrule\nNaN & 6544 \\textcolor{gray}{[5988, 7100]} & \\texttt{Ant-v4} \\\\\nNaN & 10461 \\textcolor{gray}{[10081, 10841]} & \\texttt{HalfCheetah-v4} \\\\\nNaN & 3101 \\textcolor{gray}{[2474, 3728]} & \\texttt{Hopper-v4} \\\\\nNaN & 7654 \\textcolor{gray}{[6724, 8583]} & \\texttt{Humanoid-v4} \\\\\nNaN & 4685 \\textcolor{gray}{[4513, 4858]} & \\texttt{Walker2d-v4} \\\\\nIQM & 1.219 \\textcolor{gray}{[1.114, 1.35]} & NaN \\\\\nMedian & 1.289 \\textcolor{gray}{[1.159, 1.363]} & NaN \\\\\nMean & 1.256 \\textcolor{gray}{[1.161, 1.349]} & NaN \\\\\n\\bottomrule\n\\end{tabular}\n'

In [26]:
with open('simba_mujoco_results.tex', 'w') as tf:
    tf.write(mujoco_latex_df)

### DMC Easy & MuJoCo (BRO rr2)

In [29]:
bro_eval_df = pd.read_csv('/home/nas4_user/youngdolee/SimbaV2_dev_imp/results/baseline/bro_utd2_dmc1m+mujoco1m.csv', index_col=0)
bro_eval_df['exp_name'] = 'BRO'
bro_eval_df['env_name'] = bro_eval_df['env_name'].str.replace('_', '-')
bro_dmc_eval_df = bro_eval_df[bro_eval_df['env_name'].isin(DMC_EASY_MEDIUM)].sort_values("env_name")
bro_mujoco_eval_df = bro_eval_df[bro_eval_df['env_name'].isin(MUJOCO_ALL)].sort_values("env_name")
total_env_steps = 1e6
metric = "avg_return"

In [46]:
experiments = bro_dmc_eval_df["exp_name"].unique()
environments = bro_dmc_eval_df["env_name"].unique()

bro_dmc_eval_df["value"] = bro_dmc_eval_df["value"] / 1000
metric_matrix_dict = generate_metric_matrix_dict(
    bro_dmc_eval_df, 
    env_step=total_env_steps, 
    metric_type=metric,
)
bro_dmc_eval_df["value"] = bro_dmc_eval_df["value"] * 1000.0
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    metric_matrix_dict, aggregate_func, reps=10000
)
print(aggregate_scores)

bro_dmc_eval_df = bro_dmc_eval_df[bro_dmc_eval_df["metric"] == metric]
bro_dmc_eval_df = bro_dmc_eval_df[bro_dmc_eval_df["env_step"] <= total_env_steps]

bro_dmc_em_full_results_df = pd.DataFrame(columns=["Task", "BRO"])

for _exp_name in experiments:
    _exp_data = bro_dmc_eval_df[bro_dmc_eval_df["exp_name"] == _exp_name]
    _num_seeds = _exp_data["seed"].nunique()
    print(f"exp_name: {_exp_name} - num_seeds: {_num_seeds}")
    
for i, env in enumerate(environments):
    mean_ci = []
    env_data = bro_dmc_eval_df[bro_dmc_eval_df["env_name"] == env]
    print(env, env_data["seed"].nunique())
    for j, exp in enumerate(experiments):
        exp_data = env_data[env_data["exp_name"] == exp]
        if len(exp_data) == 0:
            continue
        assert max(exp_data["env_step"]) == total_env_steps
        exp_data = exp_data[exp_data["env_step"] == total_env_steps]
        num_seeds = exp_data["seed"].nunique()
        
        grouped_data = exp_data.groupby("env_step")["value"]

        env_steps = grouped_data.mean().index.values
        mean = float(grouped_data.mean().values)
        std_err = float(grouped_data.sem().values)

        # Plot mean history
        low_CI = mean - 1.960 * std_err
        high_CI = mean + 1.960 * std_err
        
        if metric == "avg_success":
            mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
        else:    
            mean, low_CI, high_CI = int(mean), int(low_CI), int(high_CI)
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    bro_dmc_em_full_results_df = pd.concat([bro_dmc_em_full_results_df, pd.DataFrame.from_dict(
                            data=[{"Task": "\\texttt{" + str(env) + "}",
                            "BRO": mean_ci[0],
                            }], orient='columns')], ignore_index=True)


for i, agg in enumerate(["IQM", "Median", "Mean"]):
    mean_ci = []
    for j, exp in enumerate(experiments):
        mean = aggregate_scores[exp][i]
        low_CI = aggregate_score_cis[exp][0][i]
        high_CI = aggregate_score_cis[exp][1][i]
        
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)
        
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    bro_dmc_em_full_results_df = pd.concat([bro_dmc_em_full_results_df, pd.DataFrame.from_dict(
                        data=[{"Task": agg,
                            "BRO": mean_ci[0],
                        }], orient='columns')], ignore_index=True)

bro_dmc_em_full_results_df

{'BRO': array([0.92781355, 0.87192706, 0.86109668])}
exp_name: BRO - num_seeds: 10
acrobot-swingup 5
ball-in-cup-catch 5
cartpole-balance 5
cartpole-balance-sparse 10
cartpole-swingup 5
cartpole-swingup-sparse 10
cheetah-run 5
finger-spin 5
finger-turn-easy 10
finger-turn-hard 10
fish-swim 5
hopper-hop 5
hopper-stand 5
pendulum-swingup 5
quadruped-run 5
quadruped-walk 5
reacher-easy 5
reacher-hard 5
walker-run 5
walker-stand 5
walker-walk 5


,Task,BRO
0,\texttt{acrobot-swingup},"529 \textcolor{gray}{[504, 555]}"
1,\texttt{ball-in-cup-catch},"982 \textcolor{gray}{[981, 984]}"
2,\texttt{cartpole-balance},"999 \textcolor{gray}{[998, 999]}"
3,\texttt{cartpole-balance-sparse},"852 \textcolor{gray}{[563, 1141]}"
4,\texttt{cartpole-swingup},"879 \textcolor{gray}{[877, 882]}"
5,\texttt{cartpole-swingup-sparse},"840 \textcolor{gray}{[827, 852]}"
6,\texttt{cheetah-run},"863 \textcolor{gray}{[822, 904]}"
7,\texttt{finger-spin},"988 \textcolor{gray}{[987, 989]}"
8,\texttt{finger-turn-easy},"957 \textcolor{gray}{[923, 992]}"
9,\texttt{finger-turn-hard},"957 \textcolor{gray}{[920, 993]}"


In [31]:
bro_dmc_em_latex_df = bro_dmc_em_full_results_df.to_latex(index=False)
bro_dmc_em_latex_df

'\\begin{tabular}{ll}\n\\toprule\nTask & BRO \\\\\n\\midrule\n\\texttt{acrobot-swingup} & 529 \\textcolor{gray}{[504, 555]} \\\\\n\\texttt{ball-in-cup-catch} & 982 \\textcolor{gray}{[981, 984]} \\\\\n\\texttt{cartpole-balance} & 999 \\textcolor{gray}{[998, 999]} \\\\\n\\texttt{cartpole-balance-sparse} & 852 \\textcolor{gray}{[563, 1141]} \\\\\n\\texttt{cartpole-swingup} & 879 \\textcolor{gray}{[877, 882]} \\\\\n\\texttt{cartpole-swingup-sparse} & 840 \\textcolor{gray}{[827, 852]} \\\\\n\\texttt{cheetah-run} & 863 \\textcolor{gray}{[822, 904]} \\\\\n\\texttt{finger-spin} & 988 \\textcolor{gray}{[987, 989]} \\\\\n\\texttt{finger-turn-easy} & 957 \\textcolor{gray}{[923, 992]} \\\\\n\\texttt{finger-turn-hard} & 957 \\textcolor{gray}{[920, 993]} \\\\\n\\texttt{fish-swim} & 618 \\textcolor{gray}{[523, 713]} \\\\\n\\texttt{hopper-hop} & 295 \\textcolor{gray}{[273, 316]} \\\\\n\\texttt{hopper-stand} & 949 \\textcolor{gray}{[941, 957]} \\\\\n\\texttt{pendulum-swingup} & 829 \\textcolor{gray}{[7

In [32]:
with open('bro_dmc_results.tex', 'w') as tf:
    tf.write(bro_dmc_em_latex_df)

In [33]:
experiments = bro_mujoco_eval_df["exp_name"].unique()
environments = bro_mujoco_eval_df["env_name"].unique()

metric_matrix_dict = generate_metric_matrix_dict(
    normalize_score_with_random_and_base_score(
        df=bro_mujoco_eval_df, 
        random_score_dict=MUJOCO_RANDOM_SCORE,
        base_score_dict=MUJOCO_TD3_SCORE,
    ),
    env_step=total_env_steps, 
    metric_type=metric,
)
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    metric_matrix_dict, aggregate_func, reps=10000
)
print(aggregate_scores)

bro_mujoco_eval_df = bro_mujoco_eval_df[bro_mujoco_eval_df["metric"] == metric]
bro_mujoco_eval_df = bro_mujoco_eval_df[bro_mujoco_eval_df["env_step"] <= total_env_steps]

bro_mujoco_full_results_df = pd.DataFrame(columns=["Task", "BRO"])

for _exp_name in experiments:
    _exp_data = bro_mujoco_eval_df[bro_mujoco_eval_df["exp_name"] == _exp_name]
    _num_seeds = _exp_data["seed"].nunique()
    print(f"exp_name: {_exp_name} - num_seeds: {_num_seeds}")
    
for i, env in enumerate(environments):
    mean_ci = []
    env_data = bro_mujoco_eval_df[bro_mujoco_eval_df["env_name"] == env]
    for j, exp in enumerate(experiments):
        exp_data = env_data[env_data["exp_name"] == exp]
        if len(exp_data) == 0:
            continue
        assert max(exp_data["env_step"]) == total_env_steps
        exp_data = exp_data[exp_data["env_step"] == total_env_steps]
        num_seeds = exp_data["seed"].nunique()
        
        grouped_data = exp_data.groupby("env_step")["value"]

        env_steps = grouped_data.mean().index.values
        mean = float(grouped_data.mean().values)
        std_err = float(grouped_data.sem().values)

        # Plot mean history
        low_CI = mean - 1.960 * std_err
        high_CI = mean + 1.960 * std_err
        
        if metric == "avg_success":
            mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
        else:    
            mean, low_CI, high_CI = int(mean), int(low_CI), int(high_CI)
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    bro_mujoco_full_results_df = pd.concat([bro_mujoco_full_results_df, pd.DataFrame.from_dict(
                            data=[{"Task": "\\texttt{" + str(env) + "}",
                            "BRO": mean_ci[0],
                            }], orient='columns')], ignore_index=True)


for i, agg in enumerate(["IQM", "Median", "Mean"]):
    mean_ci = []
    for j, exp in enumerate(experiments):
        mean = aggregate_scores[exp][i]
        low_CI = aggregate_score_cis[exp][0][i]
        high_CI = aggregate_score_cis[exp][1][i]
        
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)
        
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    bro_mujoco_full_results_df = pd.concat([bro_mujoco_full_results_df, pd.DataFrame.from_dict(
                        data=[{"Task": agg,
                            "BRO": mean_ci[0],
                        }], orient='columns')], ignore_index=True)

bro_mujoco_full_results_df

{'BRO': array([1.07126375, 1.07108505, 1.10121   ])}
exp_name: BRO - num_seeds: 5


,Task,BRO
0,\texttt{Ant-v4},"7027 \textcolor{gray}{[6710, 7343]}"
1,\texttt{HalfCheetah-v4},"13747 \textcolor{gray}{[12621, 14873]}"
2,\texttt{Hopper-v4},"2122 \textcolor{gray}{[1655, 2588]}"
3,\texttt{Humanoid-v4},"4757 \textcolor{gray}{[3139, 6376]}"
4,\texttt{Walker2d-v4},"3432 \textcolor{gray}{[2064, 4801]}"
5,IQM,"1.071 \textcolor{gray}{[0.828, 1.333]}"
6,Median,"1.071 \textcolor{gray}{[0.884, 1.322]}"
7,Mean,"1.101 \textcolor{gray}{[0.927, 1.278]}"


In [34]:
bro_mujoco_latex_df = bro_mujoco_full_results_df.to_latex(index=False)
bro_mujoco_latex_df

'\\begin{tabular}{ll}\n\\toprule\nTask & BRO \\\\\n\\midrule\n\\texttt{Ant-v4} & 7027 \\textcolor{gray}{[6710, 7343]} \\\\\n\\texttt{HalfCheetah-v4} & 13747 \\textcolor{gray}{[12621, 14873]} \\\\\n\\texttt{Hopper-v4} & 2122 \\textcolor{gray}{[1655, 2588]} \\\\\n\\texttt{Humanoid-v4} & 4757 \\textcolor{gray}{[3139, 6376]} \\\\\n\\texttt{Walker2d-v4} & 3432 \\textcolor{gray}{[2064, 4801]} \\\\\nIQM & 1.071 \\textcolor{gray}{[0.828, 1.333]} \\\\\nMedian & 1.071 \\textcolor{gray}{[0.884, 1.322]} \\\\\nMean & 1.101 \\textcolor{gray}{[0.927, 1.278]} \\\\\n\\bottomrule\n\\end{tabular}\n'

In [35]:
with open('bro_mujoco_results.tex', 'w') as tf:
    tf.write(bro_mujoco_latex_df)

#### DMC (Simba 1M)

In [36]:
sac_simba_1m_eval_df = pd.read_csv('../../results/1219_sac_simba_1m.csv', index_col=0)
sac_simba_1m_eval_df['exp_name'] = 'Simba'
sac_simba_1m_eval_df['env_name'] = sac_simba_1m_eval_df['env_name'].str.replace('_', '-')
sac_simba_dmc_eval_df = sac_simba_1m_eval_df[sac_simba_1m_eval_df['env_name'].isin(DMC_EASY_MEDIUM)].sort_values("env_name")
sac_simba_dmc_eval_df = sac_simba_dmc_eval_df[sac_simba_dmc_eval_df["seed"].isin([0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000])]
total_env_steps = DMC_STEPS
metric = "avg_return"


In [37]:
experiments = sac_simba_dmc_eval_df["exp_name"].unique()
environments = sac_simba_dmc_eval_df["env_name"].unique()

sac_simba_dmc_eval_df["value"] = sac_simba_dmc_eval_df["value"] / 1000
metric_matrix_dict = generate_metric_matrix_dict(
    sac_simba_dmc_eval_df, 
    env_step=total_env_steps, 
    metric_type=metric,
)
sac_simba_dmc_eval_df["value"] = sac_simba_dmc_eval_df["value"] * 1000.0
aggregate_scores, aggregate_score_cis = rly.get_interval_estimates(
    metric_matrix_dict, aggregate_func, reps=10000
)
print(aggregate_scores)

sac_simba_dmc_eval_df = sac_simba_dmc_eval_df[sac_simba_dmc_eval_df["metric"] == metric]
sac_simba_dmc_eval_df = sac_simba_dmc_eval_df[sac_simba_dmc_eval_df["env_step"] <= total_env_steps]

sac_simba_dmc_full_results_df = pd.DataFrame(columns=["Task", "Simba"])

for _exp_name in experiments:
    _exp_data = sac_simba_dmc_eval_df[sac_simba_dmc_eval_df["exp_name"] == _exp_name]
    _num_seeds = _exp_data["seed"].nunique()
    print(f"exp_name: {_exp_name} - num_seeds: {_num_seeds}")
    
for i, env in enumerate(environments):
    mean_ci = []
    env_data = sac_simba_dmc_eval_df[sac_simba_dmc_eval_df["env_name"] == env]
    for j, exp in enumerate(experiments):
        exp_data = env_data[env_data["exp_name"] == exp]
        if len(exp_data) == 0:
            continue
        assert max(exp_data["env_step"]) == total_env_steps
        exp_data = exp_data[exp_data["env_step"] == total_env_steps]
        num_seeds = exp_data["seed"].nunique()
        
        grouped_data = exp_data.groupby("env_step")["value"]

        env_steps = grouped_data.mean().index.values
        mean = float(grouped_data.mean().values)
        std_err = float(grouped_data.sem().values)

        # Plot mean history
        low_CI = mean - 1.960 * std_err
        high_CI = mean + 1.960 * std_err
        
        if metric == "avg_success":
            mean, low_CI, high_CI = round(mean * 100.0, 1), round(low_CI * 100.0, 1), round(high_CI * 100.0, 1)
        else:    
            mean, low_CI, high_CI = int(mean), int(low_CI), int(high_CI)
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    sac_simba_dmc_full_results_df = pd.concat([sac_simba_dmc_full_results_df, pd.DataFrame.from_dict(
                            data=[{"Task": "\\texttt{" + str(env) + "}",
                            "Simba": mean_ci[0],
                            }], orient='columns')], ignore_index=True)


for i, agg in enumerate(["IQM", "Median", "Mean"]):
    mean_ci = []
    for j, exp in enumerate(experiments):
        mean = aggregate_scores[exp][i]
        low_CI = aggregate_score_cis[exp][0][i]
        high_CI = aggregate_score_cis[exp][1][i]
        
        mean, low_CI, high_CI = round(mean, 3), round(low_CI, 3), round(high_CI, 3)
        
        mean_ci.append(f"{mean}" + " \\textcolor{gray}{[" + f"{low_CI}, {high_CI}]" + "}")
    sac_simba_dmc_full_results_df = pd.concat([sac_simba_dmc_full_results_df, pd.DataFrame.from_dict(
                        data=[{"Task": agg,
                            "Simba": mean_ci[0],
                        }], orient='columns')], ignore_index=True)

sac_simba_dmc_full_results_df

{'Simba': array([0.92239002, 0.86956903, 0.86437827])}
exp_name: Simba - num_seeds: 10


,Task,Simba
0,\texttt{acrobot-swingup},"431 \textcolor{gray}{[379, 482]}"
1,\texttt{ball-in-cup-catch},"981 \textcolor{gray}{[978, 983]}"
2,\texttt{cartpole-balance},"998 \textcolor{gray}{[998, 999]}"
3,\texttt{cartpole-balance-sparse},"991 \textcolor{gray}{[973, 1008]}"
4,\texttt{cartpole-swingup},"876 \textcolor{gray}{[871, 881]}"
5,\texttt{cartpole-swingup-sparse},"825 \textcolor{gray}{[795, 854]}"
6,\texttt{cheetah-run},"877 \textcolor{gray}{[849, 905]}"
7,\texttt{finger-spin},"849 \textcolor{gray}{[758, 939]}"
8,\texttt{finger-turn-easy},"935 \textcolor{gray}{[903, 968]}"
9,\texttt{finger-turn-hard},"915 \textcolor{gray}{[859, 972]}"


In [38]:
sac_simba_dmc_latex_df = sac_simba_dmc_full_results_df.to_latex(index=False)
sac_simba_dmc_latex_df

'\\begin{tabular}{ll}\n\\toprule\nTask & Simba \\\\\n\\midrule\n\\texttt{acrobot-swingup} & 431 \\textcolor{gray}{[379, 482]} \\\\\n\\texttt{ball-in-cup-catch} & 981 \\textcolor{gray}{[978, 983]} \\\\\n\\texttt{cartpole-balance} & 998 \\textcolor{gray}{[998, 999]} \\\\\n\\texttt{cartpole-balance-sparse} & 991 \\textcolor{gray}{[973, 1008]} \\\\\n\\texttt{cartpole-swingup} & 876 \\textcolor{gray}{[871, 881]} \\\\\n\\texttt{cartpole-swingup-sparse} & 825 \\textcolor{gray}{[795, 854]} \\\\\n\\texttt{cheetah-run} & 877 \\textcolor{gray}{[849, 905]} \\\\\n\\texttt{finger-spin} & 849 \\textcolor{gray}{[758, 939]} \\\\\n\\texttt{finger-turn-easy} & 935 \\textcolor{gray}{[903, 968]} \\\\\n\\texttt{finger-turn-hard} & 915 \\textcolor{gray}{[859, 972]} \\\\\n\\texttt{fish-swim} & 823 \\textcolor{gray}{[799, 846]} \\\\\n\\texttt{hopper-hop} & 385 \\textcolor{gray}{[322, 449]} \\\\\n\\texttt{hopper-stand} & 929 \\textcolor{gray}{[900, 957]} \\\\\n\\texttt{pendulum-swingup} & 737 \\textcolor{gray}{

In [39]:
with open('simba_dmc_1m_results.tex', 'w') as tf:
    tf.write(sac_simba_dmc_latex_df)

### DMC Easy (HyperSimba)

In [40]:
hypersimba_eval_df['env_name'] = hypersimba_eval_df['env_name'].str.replace('_', '-')
dmc_easy_eval_df = hypersimba_eval_df[hypersimba_eval_df['env_name'].isin(DMC_EASY_MEDIUM)]
dmc_easy_eval_df = dmc_easy_eval_df.sort_values(by = ['env_name'])

dmc_easy_eval_df["value"] = dmc_easy_eval_df["value"] / 1000
dmc_easy_metric_matrix_dict = generate_metric_matrix_dict(
    dmc_easy_eval_df, 
    env_step=DMC_STEPS, 
    metric_type="avg_return",
)
dmc_easy_eval_df["value"] = dmc_easy_eval_df["value"] * 1000
dmc_easy_full_results_df = full_results_per_env(
    dmc_easy_eval_df,
    metric_matrix_dict=dmc_easy_metric_matrix_dict,
    total_env_steps=DMC_STEPS,
    aggregate_func=aggregate_func,
    columns=["SimbaV2"],
)
dmc_easy_full_results_df

exp_name: SimbaV2 - num_seeds: 10


,\textbf{Task},\textbf{SimbaV2}
0,\texttt{acrobot-swingup},"436 \textcolor{gray}{[391, 482]}"
1,\texttt{ball-in-cup-catch},"982 \textcolor{gray}{[980, 984]}"
2,\texttt{cartpole-balance},"999 \textcolor{gray}{[999, 999]}"
3,\texttt{cartpole-balance-sparse},"967 \textcolor{gray}{[904, 1030]}"
4,\texttt{cartpole-swingup},"880 \textcolor{gray}{[876, 883]}"
5,\texttt{cartpole-swingup-sparse},"848 \textcolor{gray}{[848, 849]}"
6,\texttt{cheetah-run},"920 \textcolor{gray}{[918, 922]}"
7,\texttt{finger-spin},"891 \textcolor{gray}{[810, 972]}"
8,\texttt{finger-turn-easy},"953 \textcolor{gray}{[925, 980]}"
9,\texttt{finger-turn-hard},"951 \textcolor{gray}{[925, 977]}"


In [41]:
dmc_easy_latex_df = dmc_easy_full_results_df.to_latex(index=False)
dmc_easy_latex_df

'\\begin{tabular}{ll}\n\\toprule\n\\textbf{Task} & \\textbf{SimbaV2} \\\\\n\\midrule\n\\texttt{acrobot-swingup} & 436 \\textcolor{gray}{[391, 482]} \\\\\n\\texttt{ball-in-cup-catch} & 982 \\textcolor{gray}{[980, 984]} \\\\\n\\texttt{cartpole-balance} & 999 \\textcolor{gray}{[999, 999]} \\\\\n\\texttt{cartpole-balance-sparse} & 967 \\textcolor{gray}{[904, 1030]} \\\\\n\\texttt{cartpole-swingup} & 880 \\textcolor{gray}{[876, 883]} \\\\\n\\texttt{cartpole-swingup-sparse} & 848 \\textcolor{gray}{[848, 849]} \\\\\n\\texttt{cheetah-run} & 920 \\textcolor{gray}{[918, 922]} \\\\\n\\texttt{finger-spin} & 891 \\textcolor{gray}{[810, 972]} \\\\\n\\texttt{finger-turn-easy} & 953 \\textcolor{gray}{[925, 980]} \\\\\n\\texttt{finger-turn-hard} & 951 \\textcolor{gray}{[925, 977]} \\\\\n\\texttt{fish-swim} & 826 \\textcolor{gray}{[806, 846]} \\\\\n\\texttt{hopper-hop} & 290 \\textcolor{gray}{[233, 348]} \\\\\n\\texttt{hopper-stand} & 944 \\textcolor{gray}{[926, 962]} \\\\\n\\texttt{pendulum-swingup} & 

In [42]:
with open('dmc_easy_results.tex', 'w') as tf:
    tf.write(dmc_easy_latex_df)

#### DMC Hard

In [57]:
# SAC, SR-SAC, iQRL, BRO, MAD-TD 
sac_as_df = pd.read_csv('../../results/baseline/sac.csv')
sac_as_df['exp_name'] = "SAC"

sr_as_df = pd.read_csv('../../results/0112_sr-sac-utd32_dmc-hard_1m.csv')
sr_as_df['exp_name'] = "SR-SAC"

iqrl_as_df = pd.read_csv('../../results/0118_iQRL_dmc_hard.csv')
iqrl_as_df['exp_name'] = "iQRL"

bro_as_df = pd.read_csv('../../results/baseline/bro.csv')
bro_as_df['exp_name'] = "BRO"

mad_as_df = pd.read_csv('../../results/0110_mad-td-utd=8-model=0.05_dmc_hard.csv')
mad_as_df['exp_name'] = "MAD-TD"
mad_as_df = mad_as_df[mad_as_df["seed"] <= 9]

_dmc_hard_df = pd.concat([
    sac_as_df, 
    sr_as_df,
    iqrl_as_df,
    bro_as_df,
    mad_as_df
], ignore_index=True, sort=False)
_dmc_hard_df['env_name'] = _dmc_hard_df['env_name'].str.replace('_', '-')
_dmc_hard_df = _dmc_hard_df[_dmc_hard_df['env_name'].isin(DMC_HARD)]
_dmc_hard_df = _dmc_hard_df.sort_values(by = ['env_name'])

FileNotFoundError: [Errno 2] No such file or directory: '../../results/baseline/bro.csv'

In [58]:
_dmc_hard_df["value"] = _dmc_hard_df["value"] / 1000
_dmc_hard_metric_matrix_dict = generate_metric_matrix_dict(
    _dmc_hard_df, 
    env_step=DMC_STEPS, 
    metric_type="avg_return",
)
_dmc_hard_df["value"] = _dmc_hard_df["value"] * 1000
_dmc_hard_full_results_df = full_results_per_env(
    _dmc_hard_df,
    metric_matrix_dict=_dmc_hard_metric_matrix_dict,
    total_env_steps=DMC_STEPS,
    aggregate_func=aggregate_func,
    columns=['SAC', 'SR-SAC', 'iQRL', 'BRO', 'MAD-TD']
)
_dmc_hard_full_results_df

exp_name: MAD-TD - num_seeds: 10
exp_name: BRO - num_seeds: 10
exp_name: SR-SAC - num_seeds: 5
exp_name: SAC - num_seeds: 10
exp_name: iQRL - num_seeds: 3


,\textbf{Task},\textbf{SAC},\textbf{SR-SAC},\textbf{iQRL},\textbf{BRO},\textbf{MAD-TD}
0,\texttt{dog-run},"36 \textcolor{gray}{[3, 69]}","6 \textcolor{gray}{[5, 8]}","380 \textcolor{gray}{[336, 424]}","374 \textcolor{gray}{[338, 411]}","437 \textcolor{gray}{[396, 478]}"
1,\texttt{dog-stand},"102 \textcolor{gray}{[39, 164]}","24 \textcolor{gray}{[17, 31]}","926 \textcolor{gray}{[897, 955]}","966 \textcolor{gray}{[956, 977]}","967 \textcolor{gray}{[952, 982]}"
2,\texttt{dog-trot},"29 \textcolor{gray}{[6, 52]}","13 \textcolor{gray}{[8, 18]}","713 \textcolor{gray}{[516, 909]}","783 \textcolor{gray}{[717, 848]}","867 \textcolor{gray}{[805, 929]}"
3,\texttt{dog-walk},"38 \textcolor{gray}{[11, 65]}","8 \textcolor{gray}{[7, 9]}","866 \textcolor{gray}{[827, 905]}","931 \textcolor{gray}{[920, 942]}","924 \textcolor{gray}{[906, 943]}"
4,\texttt{humanoid-run},"116 \textcolor{gray}{[89, 144]}","121 \textcolor{gray}{[21, 221]}","188 \textcolor{gray}{[167, 210]}","204 \textcolor{gray}{[186, 223]}","200 \textcolor{gray}{[180, 220]}"
5,\texttt{humanoid-stand},"352 \textcolor{gray}{[225, 479]}","161 \textcolor{gray}{[-36, 359]}","727 \textcolor{gray}{[655, 799]}","920 \textcolor{gray}{[909, 931]}","870 \textcolor{gray}{[840, 901]}"
6,\texttt{humanoid-walk},"273 \textcolor{gray}{[128, 418]}","122 \textcolor{gray}{[-115, 359]}","688 \textcolor{gray}{[642, 735]}","672 \textcolor{gray}{[619, 725]}","684 \textcolor{gray}{[609, 759]}"
7,IQM,"0.069 \textcolor{gray}{[0.041, 0.114]}","0.011 \textcolor{gray}{[0.007, 0.038]}","0.694 \textcolor{gray}{[0.531, 0.806]}","0.772 \textcolor{gray}{[0.666, 0.853]}","0.787 \textcolor{gray}{[0.693, 0.865]}"
8,Median,"0.159 \textcolor{gray}{[0.08, 0.183]}","0.049 \textcolor{gray}{[0.01, 0.099]}","0.64 \textcolor{gray}{[0.512, 0.765]}","0.694 \textcolor{gray}{[0.618, 0.776]}","0.707 \textcolor{gray}{[0.632, 0.788]}"
9,Mean,"0.136 \textcolor{gray}{[0.099, 0.174]}","0.065 \textcolor{gray}{[0.026, 0.116]}","0.642 \textcolor{gray}{[0.532, 0.746]}","0.693 \textcolor{gray}{[0.627, 0.758]}","0.708 \textcolor{gray}{[0.642, 0.771]}"


In [59]:
_dmc_hard_latex_df = _dmc_hard_full_results_df.to_latex(index=False)
_dmc_hard_latex_df

'\\begin{tabular}{llllll}\n\\toprule\n\\textbf{Task} & \\textbf{SAC} & \\textbf{SR-SAC} & \\textbf{iQRL} & \\textbf{BRO} & \\textbf{MAD-TD} \\\\\n\\midrule\n\\texttt{dog-run} & 36 \\textcolor{gray}{[3, 69]} & 6 \\textcolor{gray}{[5, 8]} & 380 \\textcolor{gray}{[336, 424]} & 374 \\textcolor{gray}{[338, 411]} & 437 \\textcolor{gray}{[396, 478]} \\\\\n\\texttt{dog-stand} & 102 \\textcolor{gray}{[39, 164]} & 24 \\textcolor{gray}{[17, 31]} & 926 \\textcolor{gray}{[897, 955]} & 966 \\textcolor{gray}{[956, 977]} & 967 \\textcolor{gray}{[952, 982]} \\\\\n\\texttt{dog-trot} & 29 \\textcolor{gray}{[6, 52]} & 13 \\textcolor{gray}{[8, 18]} & 713 \\textcolor{gray}{[516, 909]} & 783 \\textcolor{gray}{[717, 848]} & 867 \\textcolor{gray}{[805, 929]} \\\\\n\\texttt{dog-walk} & 38 \\textcolor{gray}{[11, 65]} & 8 \\textcolor{gray}{[7, 9]} & 866 \\textcolor{gray}{[827, 905]} & 931 \\textcolor{gray}{[920, 942]} & 924 \\textcolor{gray}{[906, 943]} \\\\\n\\texttt{humanoid-run} & 116 \\textcolor{gray}{[89, 14

In [60]:
with open('dmc_hard_results_2.tex', 'w') as tf:
    tf.write(_dmc_hard_latex_df)

In [61]:
dmc_hard_eval_df = off_policy_df[off_policy_df['env_name'].isin(DMC_HARD)]

dmc_hard_eval_df["value"] = dmc_hard_eval_df["value"] / 1000
dmc_hard_metric_matrix_dict = generate_metric_matrix_dict(
    dmc_hard_eval_df, 
    env_step=DMC_STEPS, 
    metric_type="avg_return",
)
dmc_hard_eval_df["value"] = dmc_hard_eval_df["value"] * 1000
dmc_hard_full_results_df = full_results_per_env(
    dmc_hard_eval_df,
    metric_matrix_dict=dmc_hard_metric_matrix_dict,
    total_env_steps=DMC_STEPS,
    aggregate_func=aggregate_func,
    columns = ["SimbaV2", "Simba", "TD-MPC2", "DreamerV3", "TD7"],
)
dmc_hard_full_results_df

/tmp/ipykernel_1977766/1065301516.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dmc_hard_eval_df["value"] = dmc_hard_eval_df["value"] / 1000
/tmp/ipykernel_1977766/1065301516.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dmc_hard_eval_df["value"] = dmc_hard_eval_df["value"] * 1000


exp_name: SimbaV2 - num_seeds: 10
exp_name: Simba - num_seeds: 15
exp_name: TD7 - num_seeds: 5
exp_name: TD-MPC2 - num_seeds: 3
exp_name: DreamerV3 - num_seeds: 3


,\textbf{Task},\textbf{SimbaV2},\textbf{Simba},\textbf{TD-MPC2},\textbf{DreamerV3},\textbf{TD7}
0,\texttt{dog-trot},"861 \textcolor{gray}{[772, 950]}","824 \textcolor{gray}{[773, 876]}","500 \textcolor{gray}{[481, 518]}","10 \textcolor{gray}{[6, 14]}","125 \textcolor{gray}{[43, 208]}"
1,\texttt{dog-run},"562 \textcolor{gray}{[516, 608]}","544 \textcolor{gray}{[525, 564]}","169 \textcolor{gray}{[29, 309]}","15 \textcolor{gray}{[5, 25]}","127 \textcolor{gray}{[77, 177]}"
2,\texttt{dog-walk},"935 \textcolor{gray}{[927, 944]}","916 \textcolor{gray}{[905, 928]}","493 \textcolor{gray}{[62, 925]}","23 \textcolor{gray}{[10, 35]}","280 \textcolor{gray}{[28, 532]}"
3,\texttt{dog-stand},"981 \textcolor{gray}{[977, 985]}","960 \textcolor{gray}{[951, 969]}","798 \textcolor{gray}{[766, 831]}","55 \textcolor{gray}{[34, 77]}","753 \textcolor{gray}{[666, 840]}"
4,\texttt{humanoid-run},"194 \textcolor{gray}{[182, 207]}","181 \textcolor{gray}{[171, 191]}","184 \textcolor{gray}{[157, 211]}","0 \textcolor{gray}{[0, 1]}","79 \textcolor{gray}{[16, 142]}"
5,\texttt{humanoid-walk},"651 \textcolor{gray}{[590, 713]}","668 \textcolor{gray}{[608, 728]}","628 \textcolor{gray}{[609, 646]}","1 \textcolor{gray}{[0, 1]}","252 \textcolor{gray}{[43, 462]}"
6,\texttt{humanoid-stand},"916 \textcolor{gray}{[886, 945]}","846 \textcolor{gray}{[801, 890]}","663 \textcolor{gray}{[626, 700]}","5 \textcolor{gray}{[2, 7]}","389 \textcolor{gray}{[65, 713]}"
7,IQM,"0.808 \textcolor{gray}{[0.727, 0.88]}","0.773 \textcolor{gray}{[0.712, 0.829]}","0.527 \textcolor{gray}{[0.368, 0.652]}","0.01 \textcolor{gray}{[0.004, 0.019]}","0.216 \textcolor{gray}{[0.112, 0.358]}"
8,Median,"0.729 \textcolor{gray}{[0.656, 0.807]}","0.706 \textcolor{gray}{[0.646, 0.772]}","0.528 \textcolor{gray}{[0.372, 0.622]}","0.017 \textcolor{gray}{[0.007, 0.027]}","0.273 \textcolor{gray}{[0.164, 0.406]}"
9,Mean,"0.729 \textcolor{gray}{[0.664, 0.791]}","0.706 \textcolor{gray}{[0.655, 0.753]}","0.491 \textcolor{gray}{[0.385, 0.594]}","0.016 \textcolor{gray}{[0.009, 0.025]}","0.287 \textcolor{gray}{[0.199, 0.383]}"


In [62]:
dmc_hard_latex_df = dmc_hard_full_results_df.to_latex(index=False)
dmc_hard_latex_df

'\\begin{tabular}{llllll}\n\\toprule\n\\textbf{Task} & \\textbf{SimbaV2} & \\textbf{Simba} & \\textbf{TD-MPC2} & \\textbf{DreamerV3} & \\textbf{TD7} \\\\\n\\midrule\n\\texttt{dog-trot} & 861 \\textcolor{gray}{[772, 950]} & 824 \\textcolor{gray}{[773, 876]} & 500 \\textcolor{gray}{[481, 518]} & 10 \\textcolor{gray}{[6, 14]} & 125 \\textcolor{gray}{[43, 208]} \\\\\n\\texttt{dog-run} & 562 \\textcolor{gray}{[516, 608]} & 544 \\textcolor{gray}{[525, 564]} & 169 \\textcolor{gray}{[29, 309]} & 15 \\textcolor{gray}{[5, 25]} & 127 \\textcolor{gray}{[77, 177]} \\\\\n\\texttt{dog-walk} & 935 \\textcolor{gray}{[927, 944]} & 916 \\textcolor{gray}{[905, 928]} & 493 \\textcolor{gray}{[62, 925]} & 23 \\textcolor{gray}{[10, 35]} & 280 \\textcolor{gray}{[28, 532]} \\\\\n\\texttt{dog-stand} & 981 \\textcolor{gray}{[977, 985]} & 960 \\textcolor{gray}{[951, 969]} & 798 \\textcolor{gray}{[766, 831]} & 55 \\textcolor{gray}{[34, 77]} & 753 \\textcolor{gray}{[666, 840]} \\\\\n\\texttt{humanoid-run} & 194 \\te

In [63]:
with open('dmc_hard_results.tex', 'w') as tf:
    tf.write(dmc_hard_latex_df)

#### MyoSuite

In [64]:
myosuite_eval_df = off_policy_df[off_policy_df['env_name'].isin(MYOSUITE_TASKS)]
myosuite_metric_matrix_dict = generate_metric_matrix_dict(
    myosuite_eval_df, 
    env_step=MYO_STEPS, 
    metric_type="avg_success",
)

myosuite_full_results_df = full_results_per_env(
    myosuite_eval_df,
    metric_matrix_dict=myosuite_metric_matrix_dict,
    total_env_steps=MYO_STEPS,
    aggregate_func=aggregate_func,
    metric="avg_success",
    columns=["DreamerV3", "TD7", "TD-MPC2", "Simba", "SimbaV2"],
)
myosuite_full_results_df

exp_name: SimbaV2 - num_seeds: 10
exp_name: Simba - num_seeds: 10
exp_name: TD7 - num_seeds: 5
exp_name: TD-MPC2 - num_seeds: 3
exp_name: DreamerV3 - num_seeds: 3


,\textbf{Task},\textbf{DreamerV3},\textbf{TD7},\textbf{TD-MPC2},\textbf{Simba},\textbf{SimbaV2}
0,\texttt{myo-pen-twirl-hard},"53.3 \textcolor{gray}{[29.8, 76.9]}","12.0 \textcolor{gray}{[2.4, 21.6]}","40.0 \textcolor{gray}{[40.0, 40.0]}","77.0 \textcolor{gray}{[66.4, 87.6]}","93.0 \textcolor{gray}{[88.8, 97.2]}"
1,\texttt{myo-pen-twirl},"96.7 \textcolor{gray}{[90.1, 103.2]}","100.0 \textcolor{gray}{[100.0, 100.0]}","70.0 \textcolor{gray}{[11.2, 128.8]}","80.0 \textcolor{gray}{[53.9, 106.1]}","100.0 \textcolor{gray}{[100.0, 100.0]}"
2,\texttt{myo-key-turn-hard},"0.0 \textcolor{gray}{[0.0, 0.0]}","0.0 \textcolor{gray}{[0.0, 0.0]}","0.0 \textcolor{gray}{[0.0, 0.0]}","7.0 \textcolor{gray}{[-3.1, 17.1]}","62.0 \textcolor{gray}{[42.7, 81.3]}"
3,\texttt{myo-key-turn},"88.9 \textcolor{gray}{[67.1, 110.7]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}"
4,\texttt{myo-obj-hold-hard},"9.4 \textcolor{gray}{[8.4, 10.5]}","10.0 \textcolor{gray}{[-0.7, 20.7]}","56.7 \textcolor{gray}{[39.4, 74.0]}","96.0 \textcolor{gray}{[92.8, 99.2]}","98.0 \textcolor{gray}{[95.4, 100.6]}"
5,\texttt{myo-obj-hold},"33.3 \textcolor{gray}{[-32.0, 98.7]}","20.0 \textcolor{gray}{[-19.2, 59.2]}","100.0 \textcolor{gray}{[100.0, 100.0]}","90.0 \textcolor{gray}{[70.4, 109.6]}","100.0 \textcolor{gray}{[100.0, 100.0]}"
6,\texttt{myo-pose-hard},"0.0 \textcolor{gray}{[0.0, 0.0]}","0.0 \textcolor{gray}{[0.0, 0.0]}","0.0 \textcolor{gray}{[0.0, 0.0]}","0.0 \textcolor{gray}{[0.0, 0.0]}","0.0 \textcolor{gray}{[0.0, 0.0]}"
7,\texttt{myo-pose},"100.0 \textcolor{gray}{[100.0, 100.0]}","0.0 \textcolor{gray}{[0.0, 0.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}"
8,\texttt{myo-reach-hard},"0.0 \textcolor{gray}{[0.0, 0.0]}","14.0 \textcolor{gray}{[0.7, 27.3]}","83.3 \textcolor{gray}{[66.0, 100.6]}","93.0 \textcolor{gray}{[86.4, 99.6]}","94.0 \textcolor{gray}{[87.3, 100.7]}"
9,\texttt{myo-reach},"100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}","100.0 \textcolor{gray}{[100.0, 100.0]}"


In [65]:
myosuite_latex_df = myosuite_full_results_df.to_latex(index=False)
myosuite_latex_df

'\\begin{tabular}{llllll}\n\\toprule\n\\textbf{Task} & \\textbf{DreamerV3} & \\textbf{TD7} & \\textbf{TD-MPC2} & \\textbf{Simba} & \\textbf{SimbaV2} \\\\\n\\midrule\n\\texttt{myo-pen-twirl-hard} & 53.3 \\textcolor{gray}{[29.8, 76.9]} & 12.0 \\textcolor{gray}{[2.4, 21.6]} & 40.0 \\textcolor{gray}{[40.0, 40.0]} & 77.0 \\textcolor{gray}{[66.4, 87.6]} & 93.0 \\textcolor{gray}{[88.8, 97.2]} \\\\\n\\texttt{myo-pen-twirl} & 96.7 \\textcolor{gray}{[90.1, 103.2]} & 100.0 \\textcolor{gray}{[100.0, 100.0]} & 70.0 \\textcolor{gray}{[11.2, 128.8]} & 80.0 \\textcolor{gray}{[53.9, 106.1]} & 100.0 \\textcolor{gray}{[100.0, 100.0]} \\\\\n\\texttt{myo-key-turn-hard} & 0.0 \\textcolor{gray}{[0.0, 0.0]} & 0.0 \\textcolor{gray}{[0.0, 0.0]} & 0.0 \\textcolor{gray}{[0.0, 0.0]} & 7.0 \\textcolor{gray}{[-3.1, 17.1]} & 62.0 \\textcolor{gray}{[42.7, 81.3]} \\\\\n\\texttt{myo-key-turn} & 88.9 \\textcolor{gray}{[67.1, 110.7]} & 100.0 \\textcolor{gray}{[100.0, 100.0]} & 100.0 \\textcolor{gray}{[100.0, 100.0]} & 100

In [66]:
with open('full_myosuite.tex', 'w') as tf:
    tf.write(myosuite_latex_df)

#### Humanoid Bench

In [67]:
cqn_as_df = pd.read_csv('../../results/0110_cqn_as_hbench_1m.csv')
cqn_as_df['exp_name'] = "CQN-AS"
rows_to_copy = cqn_as_df[cqn_as_df['env_step'] == 950000].copy()
rows_to_copy['env_step'] = 1e6
cqn_as_df = pd.concat([cqn_as_df, rows_to_copy], ignore_index=True)

_hb_df = pd.concat([
    hypersimba_eval_df,
    sac_simba_eval_df,
    td7_eval_df,
    tdmpc2_eval_df,
    dreamerv3_eval_df,
    cqn_as_df,
], ignore_index=True, sort=False)
_hb_df['env_name'] = _hb_df['env_name'].str.replace('_', '-')
hb_df = _hb_df[_hb_df['env_name'].isin(HB_LOCOMOTION_NOHAND)]

In [68]:
hb_metric_matrix_dict = generate_metric_matrix_dict(
    normalize_score_with_random_and_base_score(
        df=hb_df,
        random_score_dict=HB_RANDOM_SCORE,
        base_score_dict=HB_SUCCESS_SCORE,
    ),
    env_step=HB_STEPS,
    metric_type="avg_return",
)

hb_full_results_df = full_results_per_env(
    hb_df,
    metric_matrix_dict=hb_metric_matrix_dict,
    total_env_steps=HB_STEPS,
    aggregate_func=aggregate_func,
    metric="avg_return",
    columns=["DreamerV3", "TD7", "TD-MPC2", "CQN-AS", "Simba", "SimbaV2"],
)
hb_full_results_df

exp_name: SimbaV2 - num_seeds: 10
exp_name: Simba - num_seeds: 10
exp_name: TD7 - num_seeds: 5
exp_name: TD-MPC2 - num_seeds: 3
exp_name: DreamerV3 - num_seeds: 3
exp_name: CQN-AS - num_seeds: 5


,\textbf{Task},\textbf{DreamerV3},\textbf{TD7},\textbf{TD-MPC2},\textbf{CQN-AS},\textbf{Simba},\textbf{SimbaV2}
0,\texttt{h1-pole-v0},"41 \textcolor{gray}{[28, 54]}","441 \textcolor{gray}{[320, 563]}","744 \textcolor{gray}{[609, 879]}","102 \textcolor{gray}{[94, 111]}","716 \textcolor{gray}{[667, 765]}","791 \textcolor{gray}{[785, 797]}"
1,\texttt{h1-slide-v0},"11 \textcolor{gray}{[7, 15]}","39 \textcolor{gray}{[26, 53]}","334 \textcolor{gray}{[304, 364]}","50 \textcolor{gray}{[47, 54]}","277 \textcolor{gray}{[252, 303]}","487 \textcolor{gray}{[404, 571]}"
2,\texttt{h1-stair-v0},"7 \textcolor{gray}{[2, 12]}","52 \textcolor{gray}{[31, 74]}","378 \textcolor{gray}{[108, 648]}","47 \textcolor{gray}{[39, 54]}","269 \textcolor{gray}{[153, 385]}","493 \textcolor{gray}{[467, 518]}"
3,\texttt{h1-balance-hard-v0},"11 \textcolor{gray}{[7, 15]}","79 \textcolor{gray}{[51, 107]}","31 \textcolor{gray}{[5, 56]}","46 \textcolor{gray}{[44, 49]}","75 \textcolor{gray}{[71, 80]}","143 \textcolor{gray}{[128, 157]}"
4,\texttt{h1-balance-simple-v0},"9 \textcolor{gray}{[6, 12]}","69 \textcolor{gray}{[58, 80]}","42 \textcolor{gray}{[14, 70]}","63 \textcolor{gray}{[57, 70]}","337 \textcolor{gray}{[193, 482]}","723 \textcolor{gray}{[651, 795]}"
5,\texttt{h1-sit-hard-v0},"15 \textcolor{gray}{[-4, 35]}","235 \textcolor{gray}{[154, 315]}","723 \textcolor{gray}{[660, 786]}","220 \textcolor{gray}{[143, 297]}","512 \textcolor{gray}{[354, 670]}","679 \textcolor{gray}{[548, 811]}"
6,\texttt{h1-sit-simple-v0},"19 \textcolor{gray}{[9, 28]}","874 \textcolor{gray}{[869, 879]}","790 \textcolor{gray}{[772, 809]}","692 \textcolor{gray}{[512, 872]}","833 \textcolor{gray}{[814, 853]}","875 \textcolor{gray}{[870, 880]}"
7,\texttt{h1-maze-v0},"113 \textcolor{gray}{[107, 118]}","147 \textcolor{gray}{[137, 156]}","244 \textcolor{gray}{[106, 383]}","126 \textcolor{gray}{[111, 141]}","354 \textcolor{gray}{[342, 366]}","313 \textcolor{gray}{[287, 340]}"
8,\texttt{h1-crawl-v0},"248 \textcolor{gray}{[176, 319]}","582 \textcolor{gray}{[563, 600]}","962 \textcolor{gray}{[959, 965]}","564 \textcolor{gray}{[544, 585]}","923 \textcolor{gray}{[904, 942]}","946 \textcolor{gray}{[933, 959]}"
9,\texttt{h1-hurdle-v0},"4 \textcolor{gray}{[3, 5]}","60 \textcolor{gray}{[18, 102]}","387 \textcolor{gray}{[254, 519]}","23 \textcolor{gray}{[12, 33]}","175 \textcolor{gray}{[150, 201]}","202 \textcolor{gray}{[167, 236]}"


In [69]:
hb_latex_df = hb_full_results_df.to_latex(index=False)
hb_latex_df

'\\begin{tabular}{lllllll}\n\\toprule\n\\textbf{Task} & \\textbf{DreamerV3} & \\textbf{TD7} & \\textbf{TD-MPC2} & \\textbf{CQN-AS} & \\textbf{Simba} & \\textbf{SimbaV2} \\\\\n\\midrule\n\\texttt{h1-pole-v0} & 41 \\textcolor{gray}{[28, 54]} & 441 \\textcolor{gray}{[320, 563]} & 744 \\textcolor{gray}{[609, 879]} & 102 \\textcolor{gray}{[94, 111]} & 716 \\textcolor{gray}{[667, 765]} & 791 \\textcolor{gray}{[785, 797]} \\\\\n\\texttt{h1-slide-v0} & 11 \\textcolor{gray}{[7, 15]} & 39 \\textcolor{gray}{[26, 53]} & 334 \\textcolor{gray}{[304, 364]} & 50 \\textcolor{gray}{[47, 54]} & 277 \\textcolor{gray}{[252, 303]} & 487 \\textcolor{gray}{[404, 571]} \\\\\n\\texttt{h1-stair-v0} & 7 \\textcolor{gray}{[2, 12]} & 52 \\textcolor{gray}{[31, 74]} & 378 \\textcolor{gray}{[108, 648]} & 47 \\textcolor{gray}{[39, 54]} & 269 \\textcolor{gray}{[153, 385]} & 493 \\textcolor{gray}{[467, 518]} \\\\\n\\texttt{h1-balance-hard-v0} & 11 \\textcolor{gray}{[7, 15]} & 79 \\textcolor{gray}{[51, 107]} & 31 \\textco

In [70]:
with open('full_hbench.tex', 'w') as tf:
    tf.write(hb_latex_df)